In [4]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# ============================================================
# NOTEBOOK: nb_nyc_taxi_data_quality
# PURPOSE: Build Data Quality tables from yellow_taxi_silver
# TABLES:
#   1. dq_summary
#   2. dq_exceptions
#   3. dq_rule_counts
#   4. dq_exception_rule_bridge
#   5. dim_dq_rule
# ============================================================

# ------------------------------------------------------------
# 1. LOAD SILVER
# ------------------------------------------------------------

silver_df = spark.table("yellow_taxi_silver")

total_rows = silver_df.count()

pickup = F.col("tpep_pickup_datetime")
dropoff = F.col("tpep_dropoff_datetime")

duration_minutes = (
    (F.unix_timestamp(dropoff) - F.unix_timestamp(pickup)) / 60.0
)

print(f"Silver rows: {total_rows:,}")


# ------------------------------------------------------------
# 2. DEFINE DATA QUALITY RULES
# ------------------------------------------------------------

rule_data = [
    (
        "NEGATIVE_FARE",
        "Negative fare amount",
        "Review",
        "Fare amount is negative; may represent a refund or correction.",
        F.col("fare_amount") < 0
    ),
    (
        "NEGATIVE_TOTAL",
        "Negative total amount",
        "Review",
        "Total amount is negative; investigate potential financial adjustments.",
        F.col("total_amount") < 0
    ),
    (
        "NEGATIVE_TIP",
        "Negative tip amount",
        "Review",
        "Tip amount is negative; validate the transaction.",
        F.col("tip_amount") < 0
    ),
    (
        "NEGATIVE_DISTANCE",
        "Negative trip distance",
        "High",
        "Trip distance is negative and requires investigation.",
        F.col("trip_distance") < 0
    ),
    (
        "ZERO_DISTANCE",
        "Zero trip distance",
        "Review",
        "Trip distance is zero; may be a valid edge case or anomaly.",
        F.col("trip_distance") == 0
    ),
    (
        "NULL_FARE",
        "Missing fare amount",
        "High",
        "Fare amount is missing.",
        F.col("fare_amount").isNull()
    ),
    (
        "NULL_TOTAL",
        "Missing total amount",
        "High",
        "Total amount is missing.",
        F.col("total_amount").isNull()
    ),
    (
        "ZERO_DURATION",
        "Zero-duration trip",
        "Review",
        "Pickup and drop-off timestamps are equal.",
        pickup == dropoff
    ),
    (
        "LONG_DURATION",
        "Trip longer than 3 hours",
        "Review",
        "Trip duration exceeds three hours.",
        dropoff > pickup + F.expr("INTERVAL 3 HOURS")
    )
]


# ------------------------------------------------------------
# 3. CREATE dim_dq_rule
# ------------------------------------------------------------

rule_schema = T.StructType([
    T.StructField("rule_id", T.StringType(), False),
    T.StructField("rule_name", T.StringType(), False),
    T.StructField("severity", T.StringType(), False),
    T.StructField("description", T.StringType(), False)
])

dim_dq_rule_df = spark.createDataFrame(
    [
        (rule_id, rule_name, severity, description)
        for rule_id, rule_name, severity, description, condition in rule_data
    ],
    schema=rule_schema
)

(
    dim_dq_rule_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dim_dq_rule")
)

print("dim_dq_rule created:", dim_dq_rule_df.count(), "rules")


# ------------------------------------------------------------
# 4. CALCULATE RULE COUNTS
# ------------------------------------------------------------

metrics = (
    silver_df
    .agg(
        *[
            F.sum(
                F.when(condition, 1).otherwise(0)
            ).cast("long").alias(rule_id)
            for rule_id, rule_name, severity, description, condition in rule_data
        ]
    )
    .first()
    .asDict()
)

summary_rows = []

for rule_id, rule_name, severity, description, condition in rule_data:
    issue_count = int(metrics.get(rule_id) or 0)
    issue_percentage = (
        float(issue_count) / total_rows * 100.0
        if total_rows > 0 else 0.0
    )

    summary_rows.append((
        rule_id,
        rule_name,
        str(condition),
        issue_count,
        issue_percentage,
        severity,
        description,
        total_rows
    ))

summary_schema = T.StructType([
    T.StructField("rule_id", T.StringType(), False),
    T.StructField("rule_name", T.StringType(), False),
    T.StructField("rule_condition", T.StringType(), False),
    T.StructField("issue_count", T.LongType(), False),
    T.StructField("issue_percentage", T.DoubleType(), False),
    T.StructField("severity", T.StringType(), False),
    T.StructField("interpretation", T.StringType(), False),
    T.StructField("total_rows", T.LongType(), False)
])

dq_summary_df = spark.createDataFrame(
    summary_rows,
    schema=summary_schema
)

(
    dq_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dq_summary")
)

print("dq_summary created:", dq_summary_df.count(), "rules")


# ------------------------------------------------------------
# 5. CREATE DQ EXCEPTIONS
# ------------------------------------------------------------

exception_conditions = [
    (F.col("fare_amount") < 0, "NEGATIVE_FARE"),
    (F.col("total_amount") < 0, "NEGATIVE_TOTAL"),
    (F.col("tip_amount") < 0, "NEGATIVE_TIP"),
    (F.col("trip_distance") < 0, "NEGATIVE_DISTANCE"),
    (F.col("trip_distance") == 0, "ZERO_DISTANCE"),
    (F.col("fare_amount").isNull(), "NULL_FARE"),
    (F.col("total_amount").isNull(), "NULL_TOTAL"),
    (pickup == dropoff, "ZERO_DURATION"),
    (dropoff > pickup + F.expr("INTERVAL 3 HOURS"), "LONG_DURATION")
]

exception_rules_expr = F.concat_ws(
    ",",
    *[
        F.when(condition, F.lit(rule_id))
        for condition, rule_id in exception_conditions
    ]
)

exceptions_df = (
    silver_df
    .withColumn("duration_minutes", duration_minutes)
    .withColumn("exception_rules", exception_rules_expr)
    .filter(F.col("exception_rules") != "")
    .select(
        "source_file",
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "payment_type",
        "trip_distance",
        "duration_minutes",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "exception_rules"
    )
)

(
    exceptions_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dq_exceptions")
)

print("dq_exceptions created")


# ------------------------------------------------------------
# 6. CREATE INDIVIDUAL RULE COUNTS
# Include all rules, even when issue_count = 0
# ------------------------------------------------------------

dq_rule_counts_df = (
    dq_summary_df
    .select(
        "rule_id",
        F.col("issue_count").cast("long").alias("issue_count")
    )
)

(
    dq_rule_counts_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dq_rule_counts")
)

print("dq_rule_counts created:", dq_rule_counts_df.count(), "rules")

display(
    spark.table("dq_rule_counts").orderBy("rule_id")
)

# ------------------------------------------------------------
# 7. CREATE EXCEPTION-RULE BRIDGE
# ------------------------------------------------------------

dq_exception_rule_bridge_df = (
    exceptions_df
    .select(
        "source_file",
        "VendorID",
        "tpep_pickup_datetime",
        "PULocationID",
        "DOLocationID",
        F.explode(
            F.split(F.col("exception_rules"), ",")
        ).alias("rule_id")
    )
    .filter(F.col("rule_id") != "")
)

(
    dq_exception_rule_bridge_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dq_exception_rule_bridge")
)

print("dq_exception_rule_bridge created")


# ------------------------------------------------------------
# 8. VALIDATE OUTPUT TABLES
# ------------------------------------------------------------

table_names = [
    "dim_dq_rule",
    "dq_summary",
    "dq_exceptions",
    "dq_rule_counts",
    "dq_exception_rule_bridge"
]

for table_name in table_names:
    row_count = spark.table(table_name).count()
    print(f"{table_name}: {row_count:,} rows")

print("Data Quality notebook completed successfully.")

StatementMeta(, 158b466b-d61c-4a71-a8d1-df3d99f72741, 6, Finished, Available, Finished, False)

Silver rows: 78,423,712
dim_dq_rule created: 9 rules
dq_summary created: 9 rules
dq_exceptions created
dq_rule_counts created: 9 rules


SynapseWidget(Synapse.DataFrame, 6ba0835c-b62b-4535-9186-19d3328108e1)

dq_exception_rule_bridge created
dim_dq_rule: 9 rows
dq_summary: 9 rows
dq_exceptions: 6,023,630 rows
dq_rule_counts: 9 rows
dq_exception_rule_bridge: 7,443,414 rows
Data Quality notebook completed successfully.


In [3]:
from pyspark.sql import functions as F

# 1. Compare summary counts with individual rule counts
summary_check = (
    spark.table("dq_summary")
    .select(
        "rule_id",
        F.col("issue_count").alias("summary_count")
    )
)

counts_check = (
    spark.table("dq_rule_counts")
    .select(
        "rule_id",
        F.col("issue_count").alias("rule_count")
    )
)

print("=== Summary vs rule counts ===")

display(
    summary_check
    .join(counts_check, "rule_id", "full")
    .withColumn(
        "validation_status",
        F.when(
            F.col("summary_count").eqNullSafe(F.col("rule_count")),
            "MATCH"
        ).otherwise("MISMATCH")
    )
    .orderBy("rule_id")
)

# 2. Validate rule IDs against the dimension table
dim_rules = spark.table("dim_dq_rule").select("rule_id").distinct()

summary_orphans = (
    spark.table("dq_summary")
    .select("rule_id").distinct()
    .join(dim_rules, "rule_id", "left_anti")
)

counts_orphans = (
    spark.table("dq_rule_counts")
    .select("rule_id").distinct()
    .join(dim_rules, "rule_id", "left_anti")
)

bridge_orphans = (
    spark.table("dq_exception_rule_bridge")
    .select("rule_id").distinct()
    .join(dim_rules, "rule_id", "left_anti")
)

print("=== Orphan rule IDs ===")
print("dq_summary:", summary_orphans.count())
print("dq_rule_counts:", counts_orphans.count())
print("dq_exception_rule_bridge:", bridge_orphans.count())

StatementMeta(, 158b466b-d61c-4a71-a8d1-df3d99f72741, 5, Finished, Available, Finished, False)

=== Summary vs rule counts ===


SynapseWidget(Synapse.DataFrame, c1064183-ff5a-48c2-a039-59d6e7991a08)

=== Orphan rule IDs ===
dq_summary: 0
dq_rule_counts: 0
dq_exception_rule_bridge: 0


In [5]:
from pyspark.sql import functions as F

summary_check = (
    spark.table("dq_summary")
    .select(
        "rule_id",
        F.col("issue_count").alias("summary_count")
    )
)

counts_check = (
    spark.table("dq_rule_counts")
    .select(
        "rule_id",
        F.col("issue_count").alias("rule_count")
    )
)

validation = (
    summary_check
    .join(counts_check, "rule_id", "full")
    .withColumn(
        "validation_status",
        F.when(
            F.col("summary_count").eqNullSafe(F.col("rule_count")),
            "MATCH"
        ).otherwise("MISMATCH")
    )
    .orderBy("rule_id")
)

display(validation)

StatementMeta(, 158b466b-d61c-4a71-a8d1-df3d99f72741, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2955c5e3-49cc-4715-8b2e-3392492ccf3e)